In [ ]:
%load_ext autoreload
%autoreload 2

# 04 — Statement features

17 features summarising each applicant's 6-month statement. Built by `scripts/build_features.py`; this notebook checks them and measures how much signal each carries.

In [2]:
import pandas as pd
from sklearn.metrics import roc_auc_score

from mkopoguard import config
from mkopoguard.features.transactions import FEATURES
from mkopoguard.viz import apply_style, ranked_bars, save

apply_style()
table = pd.read_parquet(config.DATA_PROCESSED / "features.parquet")
holders = table[table["has_statement"] == 1]
print(f"{len(table):,} applicants, {len(holders):,} with a statement")
holders[FEATURES].describe().T[["mean", "50%", "min", "max"]].round(3)

18,718 applicants, 9,848 with a statement


,mean,50%,min,max
stmt_avg_monthly_inflow,19438.136,14123.667,533.333,257785.500
stmt_median_monthly_inflow,17587.724,12187.500,0.000,260265.500
stmt_inflow_volatility,0.366,0.263,0.006,1.501
stmt_income_payments_per_month,7.137,1.000,0.333,18.833
stmt_days_since_last_income,12.169,4.795,0.001,89.987
stmt_p2p_in_share,0.071,0.076,0.000,0.308
stmt_betting_share,0.046,0.000,0.000,0.389
stmt_airtime_share,0.100,0.068,0.004,0.582
stmt_cash_out_share,0.655,0.653,0.250,0.992
stmt_paybill_months,1.522,0.000,0.000,6.000


In [3]:
auc = pd.Series({f: roc_auc_score(holders["defaulted"], holders[f]) for f in FEATURES})
strength = auc.where(auc >= 0.5, 1 - auc)  # 0.40 is as useful as 0.60, just reversed
direction = pd.Series("higher = riskier", index=FEATURES).where(auc >= 0.5, "lower = riskier")
labels = {f: f.removeprefix("stmt_").replace("_", " ") for f in FEATURES}

fig = ranked_bars(
    strength,
    "Betting and overdraft habits carry the most signal",
    "Each statement feature on its own: ROC-AUC for predicting default (0.5 = no signal)",
    highlight=None,
    labels=labels,
    percent=False,
    baseline=0.5,
)
save(fig, config.PROJECT_ROOT / "docs" / "img" / "features_01_signal_strength.png")
pd.DataFrame({"auc": strength.round(3), "direction": direction}).sort_values("auc", ascending=False)

,auc,direction
stmt_betting_share,0.653,higher = riskier
stmt_transactions_per_month,0.598,higher = riskier
stmt_overdraft_repay_ratio,0.587,lower = riskier
stmt_overdraft_outstanding_months,0.582,higher = riskier
stmt_overdraft_borrow_count,0.578,higher = riskier
stmt_low_balance_share,0.571,higher = riskier
stmt_cash_out_share,0.568,lower = riskier
stmt_loan_to_inflow,0.557,higher = riskier
stmt_paybill_months,0.549,lower = riskier
stmt_income_payments_per_month,0.548,higher = riskier


**Takeaway:** betting share is the strongest single signal (0.653), followed by overdraft behaviour (0.58–0.59). Every feature points the sensible way. Two notes:
- *Days since last income* is lower for riskier applicants: irregular earners are paid every few days, and irregular income is riskier.
- *Money from family/friends* has no signal (0.502), as designed: remittances aren't in the default rule, so the model should learn to ignore it.

In [4]:
pairs = holders[FEATURES].corr().abs().stack()
pairs = pairs[pairs.index.get_level_values(0) < pairs.index.get_level_values(1)]
pairs.sort_values(ascending=False).head(5).round(2)

stmt_avg_monthly_inflow            stmt_median_monthly_inflow           0.92
stmt_overdraft_outstanding_months  stmt_overdraft_repay_ratio           0.82
stmt_overdraft_borrow_count        stmt_overdraft_outstanding_months    0.74
stmt_low_balance_share             stmt_overdraft_borrow_count          0.73
                                   stmt_overdraft_outstanding_months    0.73
dtype: float64

**Takeaway:** average and median monthly inflow are 0.92 correlated, and the three overdraft features 0.74–0.82. Tree models cope with overlap; for logistic regression it splits the weight between them. Phase 3 can drop one of each pair if it doesn't hurt.